In [1]:
# ===================================================
# 1. IMPORT LIBRARIES
# ===================================================
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
# ===================================================
# 2. LOAD DATA
# ===================================================
df = pd.read_csv('HealthConnect_Appointment_Data.csv')
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


In [3]:
# ===================================================
# 3. DATASET OVERVIEW
# ===================================================

# Shape
print("Rows, Columns:", df.shape)

# Column names and data types
df.info()


Rows, Columns: (5000, 18)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 no

In [4]:
# ===================================================
# 4. MISSING VALUES
# ===================================================
print("Missing values per column:")
print(df.isnull().sum())

print("\nMissing values (%):")
print((df.isnull().sum() / len(df) * 100).round(2))

Missing values per column:
appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

Missing values (%):
appointment_id            0.00
patient_id                0.00
gender                    0.00
age                       0.00
age_group                 0.00
appointment_type          0.00
booking_date              0.00
appointment_date          0.00
appointment_day           0.00
appointment_time          0.00
booking_lead_days         0.00
previous_appointments     0.00
previous_no_shows         0

In [5]:
# ===================================================
# 5. DUPLICATES
# ===================================================
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate appointment_id values:", df['appointment_id'].duplicated().sum())
print("Unique patients:", df['patient_id'].nunique())
print("Total appointments:", len(df))

Fully duplicate rows: 0
Duplicate appointment_id values: 0
Unique patients: 1696
Total appointments: 5000


In [6]:
# ===================================================
# 6. INVESTIGATE THE reminder_channel MISSING VALUES
# ===================================================
# Check whether missing reminder_channel aligns with reminder_sent = 'No'
pd.crosstab(df['reminder_sent'], df['reminder_channel'].isnull())

reminder_channel,False,True
reminder_sent,,
No,0,1366
Yes,3634,0


In [7]:
# ===================================================
# 7. CHECK UNIQUE VALUES FOR EACH CATEGORICAL COLUMN
# ===================================================
categorical_cols = ['gender', 'age_group', 'appointment_type', 'appointment_day',
                    'appointment_time', 'reminder_sent', 'reminder_channel', 
                    'appointment_outcome']

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].unique())


gender:
['Female' 'Male' 'Prefer not to say']

age_group:
['35-44' '25-34' '45-54' '55-64' '65+' '18-24']

appointment_type:
['Follow-up' 'Specialist Consultation' 'General Consultation'
 'Diagnostic Test']

appointment_day:
['Tuesday' 'Friday' 'Wednesday' 'Thursday' 'Monday' 'Sunday' 'Saturday']

appointment_time:
['Afternoon' 'Morning' 'Evening']

reminder_sent:
['Yes' 'No']

reminder_channel:
['WhatsApp' 'SMS' 'Email' nan]

appointment_outcome:
['No-Show' 'Attended' 'Cancelled']


In [8]:
# ===================================================
# 8. CHECK NUMERIC RANGES FOR SANITY (no negatives, no impossible values)
# ===================================================
numeric_cols = ['age', 'booking_lead_days', 'previous_appointments', 
                'previous_no_shows', 'distance_to_clinic_km', 'waiting_time_minutes']

df[numeric_cols].describe()

,age,booking_lead_days,previous_appointments,previous_no_shows,distance_to_clinic_km,waiting_time_minutes
count,5000.000000,5000.00000,5000.000000,5000.000000,4910.000000,4940.000000
mean,48.794800,29.63860,3.013800,0.544200,10.109572,24.189676
std,18.138547,17.39936,1.741211,0.746832,6.590030,10.863184
min,18.000000,0.00000,0.000000,0.000000,0.500000,2.000000
25%,33.000000,15.00000,2.000000,0.000000,5.300000,17.000000
50%,49.000000,30.00000,3.000000,0.000000,8.700000,24.000000
75%,64.000000,45.00000,4.000000,1.000000,13.500000,32.000000
max,80.000000,60.00000,11.000000,5.000000,45.000000,68.000000


In [9]:
# ===================================================
# 9. CHECK DATE COLUMNS
# ===================================================
print("booking_date sample values:", df['booking_date'].head(5).tolist())
print("appointment_date sample values:", df['appointment_date'].head(5).tolist())

# Attempt conversion to proper datetime
df['booking_date_parsed'] = pd.to_datetime(df['booking_date'], errors='coerce')
df['appointment_date_parsed'] = pd.to_datetime(df['appointment_date'], errors='coerce')

print("\nRows where booking_date failed to parse:", df['booking_date_parsed'].isnull().sum())
print("Rows where appointment_date failed to parse:", df['appointment_date_parsed'].isnull().sum())

booking_date sample values: ['2/6/2025', '2/25/2026', '11/16/2025', '7/18/2025', '7/9/2025']
appointment_date sample values: ['2/18/2025', '2/27/2026', '12/24/2025', '8/28/2025', '8/25/2025']

Rows where booking_date failed to parse: 0
Rows where appointment_date failed to parse: 0


In [10]:
# ===================================================
# 10. APPOINTMENT OUTCOME DISTRIBUTION (the core business problem)
# ===================================================
print(df['appointment_outcome'].value_counts())
print()
print((df['appointment_outcome'].value_counts(normalize=True) * 100).round(2))

appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64


In [11]:
# ===================================================
# 11. LOGICAL CONSISTENCY CHECK
# ===================================================
# previous_no_shows should never exceed previous_appointments
inconsistent = df[df['previous_no_shows'] > df['previous_appointments']]
print("Rows where previous_no_shows > previous_appointments:", len(inconsistent))

Rows where previous_no_shows > previous_appointments: 0
